In [ ]:
import pandas as pd
import numpy as np

DATA = '../data/individual/processed'

# load datasets
baseline_file_path = f'{DATA}/sed.csv'
session_01_file_path = f'{DATA}/sed_01.csv'
session_02_file_path = f'{DATA}/sed_02.csv'
session_03_file_path = f'{DATA}/sed_03.csv'

baseline_data = pd.read_csv(baseline_file_path)
session_01_data = pd.read_csv(session_01_file_path)
session_02_data = pd.read_csv(session_02_file_path)
session_03_data = pd.read_csv(session_03_file_path)

# fixation duration calc
def calculate_fixation_duration(df, velocity_threshold=0.1):
    df = df.copy()
    # gaze velocity
    df['gaze_velocity'] = np.sqrt(df['gazeDir.x'].diff()**2 +
                                  df['gazeDir.y'].diff()**2 +
                                  df['gazeDir.z'].diff()**2) / df['reltime'].diff()

    # identify fixations
    df['is_fixation'] = df['gaze_velocity'] < velocity_threshold

    # group fixations
    df['fixation_id'] = (df['is_fixation'] != df['is_fixation'].shift()).cumsum()

    # fixation durations
    fixation_durations = df[df['is_fixation']].groupby('fixation_id')['reltime'].apply(lambda x: x.max() - x.min())

    # filter + average
    fixation_durations = fixation_durations[fixation_durations > 0]
    average_fixation_duration = fixation_durations.mean()

    return average_fixation_duration

# avg fixation per dataset
baseline_avg_fixation_duration = calculate_fixation_duration(baseline_data)
session_01_avg_fixation_duration = calculate_fixation_duration(session_01_data)
session_02_avg_fixation_duration = calculate_fixation_duration(session_02_data)
session_03_avg_fixation_duration = calculate_fixation_duration(session_03_data)

# convert to ms
baseline_avg_fixation_duration_ms = baseline_avg_fixation_duration * 1000
session_01_avg_fixation_duration_ms = session_01_avg_fixation_duration * 1000
session_02_avg_fixation_duration_ms = session_02_avg_fixation_duration * 1000
session_03_avg_fixation_duration_ms = session_03_avg_fixation_duration * 1000

# baseline difference
fixation_difference_01 = session_01_avg_fixation_duration_ms - baseline_avg_fixation_duration_ms
fixation_difference_02 = session_02_avg_fixation_duration_ms - baseline_avg_fixation_duration_ms
fixation_difference_03 = session_03_avg_fixation_duration_ms - baseline_avg_fixation_duration_ms

# anxiety threshold
fixation_anxiety_threshold = 250

# determine anxiety
anxiety_fixation_baseline = baseline_avg_fixation_duration_ms < fixation_anxiety_threshold
anxiety_fixation_01 = session_01_avg_fixation_duration_ms < fixation_anxiety_threshold
anxiety_fixation_02 = session_02_avg_fixation_duration_ms < fixation_anxiety_threshold
anxiety_fixation_03 = session_03_avg_fixation_duration_ms < fixation_anxiety_threshold

# display results
print(f'Baseline Average Fixation Duration: {baseline_avg_fixation_duration_ms:.2f} ms - {"Anxiety" if anxiety_fixation_baseline else "Normal"}')
print(f'Session 1 Average Fixation Duration: {session_01_avg_fixation_duration_ms:.2f} ms - {"Anxiety" if anxiety_fixation_01 else "Normal"}')
print(f'Difference from Baseline in Session 1: {fixation_difference_01:.2f} ms')
print(f'Session 2 Average Fixation Duration: {session_02_avg_fixation_duration_ms:.2f} ms - {"Anxiety" if anxiety_fixation_02 else "Normal"}')
print(f'Difference from Baseline in Session 2: {fixation_difference_02:.2f} ms')
print(f'Session 3 Average Fixation Duration: {session_03_avg_fixation_duration_ms:.2f} ms - {"Anxiety" if anxiety_fixation_03 else "Normal"}')
print(f'Difference from Baseline in Session 3: {fixation_difference_03:.2f} ms')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA = '../data/individual/processed'

# load datasets
baseline_data = pd.read_csv(f'{DATA}/sed.csv')
session_01_data = pd.read_csv(f'{DATA}/sed_01.csv')
session_02_data = pd.read_csv(f'{DATA}/sed_02.csv')
session_03_data = pd.read_csv(f'{DATA}/sed_03.csv')

# calculate fixation
def calculate_fixation_durations(df, velocity_threshold=0.1):
    df = df.copy()
    df['gaze_velocity'] = np.sqrt(df['gazeDir.x'].diff()**2 +
                                  df['gazeDir.y'].diff()**2 +
                                  df['gazeDir.z'].diff()**2) / df['reltime'].diff()
    df['is_fixation'] = df['gaze_velocity'] < velocity_threshold
    df['fixation_id'] = (df['is_fixation'] != df['is_fixation'].shift()).cumsum()
    fixation_durations = df[df['is_fixation']].groupby('fixation_id')['reltime'].apply(lambda x: x.max() - x.min())
    fixation_durations = fixation_durations[fixation_durations > 0]
    return fixation_durations

# compute fixation durations
baseline_fixation_durations = calculate_fixation_durations(baseline_data)
session_01_fixation_durations = calculate_fixation_durations(session_01_data)
session_02_fixation_durations = calculate_fixation_durations(session_02_data)
session_03_fixation_durations = calculate_fixation_durations(session_03_data)

# convert to ms
baseline_fixation_durations_ms = baseline_fixation_durations * 1000
session_01_fixation_durations_ms = session_01_fixation_durations * 1000
session_02_fixation_durations_ms = session_02_fixation_durations * 1000
session_03_fixation_durations_ms = session_03_fixation_durations * 1000

# average fixation
baseline_avg_fixation_duration_ms = baseline_fixation_durations_ms.mean()
session_01_avg_fixation_duration_ms = session_01_fixation_durations_ms.mean()
session_02_avg_fixation_duration_ms = session_02_fixation_durations_ms.mean()
session_03_avg_fixation_duration_ms = session_03_fixation_durations_ms.mean()

# bar plot
average_durations = [baseline_avg_fixation_duration_ms, session_01_avg_fixation_duration_ms, session_02_avg_fixation_duration_ms, session_03_avg_fixation_duration_ms]
labels = ['Baseline', 'Session 01', 'Session 02', 'Session 03']

plt.figure(figsize=(8, 6))
plt.bar(labels, average_durations, color=['blue', 'orange', 'green', 'red'])
plt.axhline(250, color='r', linestyle='dashed', linewidth=1, label='Anxiety Threshold (<250 ms)')
plt.ylabel('Average Fixation Duration (ms)')
plt.title('Average Fixation Duration by Dataset')
plt.legend()
plt.show()
plt.close()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

DATA = '../data/individual/processed'
PSY = '../data/individual/psychometric'

# load eye data
sed_data = pd.read_csv(f'{DATA}/sed_fix_01.csv')
sed_data['datetime'] = pd.to_datetime(sed_data['datetime'], format='%Y/%m/%d %H:%M:%S.%f')

# load psychometric
psychometric_data = pd.read_csv(f'{PSY}/Psychometric_Test_Results_01.csv')
psychometric_data['Question Start Time'] = pd.to_datetime(psychometric_data['Question Start Time'], utc=True).dt.tz_convert(None)
psychometric_data['Question Answer Time'] = pd.to_datetime(psychometric_data['Question Answer Time'], utc=True).dt.tz_convert(None)

def plot_fixation_duration_for_category(category_name, expected_questions, window_size=20):
    category_data = psychometric_data[psychometric_data['Type'] == category_name].copy()

    segments = []
    question_times = []
    for _, row in category_data.iterrows():
        mask = (sed_data['datetime'] >= row['Question Start Time']) & (sed_data['datetime'] <= row['Question Answer Time'])
        seg = sed_data.loc[mask]
        if not seg.empty:
            segments.append(seg)
        question_times.append(row['Question Answer Time'])

    category_eye_data = pd.concat(segments, ignore_index=True) if segments else pd.DataFrame()
    if len(question_times) != expected_questions:
        print(f"Warning: {category_name} has {len(question_times)} questions, expected {expected_questions}")

    category_eye_data = category_eye_data.sort_values(by='datetime').reset_index(drop=True)
    category_eye_data['smoothed_fixation_duration'] = category_eye_data['duration'].rolling(window=window_size).mean()
    category_eye_data = category_eye_data[np.isfinite(category_eye_data['smoothed_fixation_duration'])]

    plt.figure(figsize=(12, 6))
    plt.plot(category_eye_data['datetime'], category_eye_data['smoothed_fixation_duration'], label='Fixation Duration', color='b')

    for i, time in enumerate(question_times, start=1):
        if not category_eye_data.empty:
            time_diffs = (category_eye_data['datetime'] - time).abs()
            nearest_idx = time_diffs.idxmin()
            fd = category_eye_data.loc[nearest_idx, 'smoothed_fixation_duration']
            plt.scatter(category_eye_data.loc[nearest_idx, 'datetime'], fd, color='red', s=50, zorder=5)
            plt.text(category_eye_data.loc[nearest_idx, 'datetime'], fd + 0.1, f'Q{i}', fontsize=9, rotation=45, ha='right')

    plt.xlabel('Time')
    plt.ylabel('Fixation Duration')
    plt.title(f'Fixation Duration Changes During {category_name}')
    plt.xticks(rotation=45)
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    plt.close()

plot_fixation_duration_for_category('HADS', 14, window_size=20)
plot_fixation_duration_for_category('STAI-T', 20, window_size=20)
plot_fixation_duration_for_category('STAI-S', 20, window_size=20)
plot_fixation_duration_for_category('BFI', 10, window_size=20)
plot_fixation_duration_for_category('FQ', 24, window_size=20)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

DATA = '../data/individual/processed'
PSY = '../data/individual/psychometric'

sed_data = pd.read_csv(f'{DATA}/sed_fix_02.csv')
sed_data['datetime'] = pd.to_datetime(sed_data['datetime'], format='%Y/%m/%d %H:%M:%S.%f')

psychometric_data = pd.read_csv(f'{PSY}/Psychometric_Test_Results_02.csv')
psychometric_data['Question Start Time'] = pd.to_datetime(psychometric_data['Question Start Time'], utc=True).dt.tz_convert(None)
psychometric_data['Question Answer Time'] = pd.to_datetime(psychometric_data['Question Answer Time'], utc=True).dt.tz_convert(None)

def plot_fixation_duration_for_category(category_name, expected_questions, window_size=20):
    category_data = psychometric_data[psychometric_data['Type'] == category_name].copy()

    segments = []
    question_times = []
    for _, row in category_data.iterrows():
        mask = (sed_data['datetime'] >= row['Question Start Time']) & (sed_data['datetime'] <= row['Question Answer Time'])
        seg = sed_data.loc[mask]
        if not seg.empty:
            segments.append(seg)
        question_times.append(row['Question Answer Time'])

    category_eye_data = pd.concat(segments, ignore_index=True) if segments else pd.DataFrame()
    if len(question_times) != expected_questions:
        print(f"Warning: {category_name} has {len(question_times)} questions, expected {expected_questions}")

    category_eye_data = category_eye_data.sort_values(by='datetime').reset_index(drop=True)
    category_eye_data['smoothed_fixation_duration'] = category_eye_data['duration'].rolling(window=window_size).mean()
    category_eye_data = category_eye_data[np.isfinite(category_eye_data['smoothed_fixation_duration'])]

    plt.figure(figsize=(12, 6))
    plt.plot(category_eye_data['datetime'], category_eye_data['smoothed_fixation_duration'], label='Fixation Duration', color='b')

    for i, time in enumerate(question_times, start=1):
        if not category_eye_data.empty:
            time_diffs = (category_eye_data['datetime'] - time).abs()
            nearest_idx = time_diffs.idxmin()
            fd = category_eye_data.loc[nearest_idx, 'smoothed_fixation_duration']
            plt.scatter(category_eye_data.loc[nearest_idx, 'datetime'], fd, color='red', s=50, zorder=5)
            plt.text(category_eye_data.loc[nearest_idx, 'datetime'], fd + 0.1, f'Q{i}', fontsize=9, rotation=45, ha='right')

    plt.xlabel('Time')
    plt.ylabel('Fixation Duration')
    plt.title(f'Fixation Duration Changes During {category_name}')
    plt.xticks(rotation=45)
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    plt.close()

plot_fixation_duration_for_category('HADS', 14, window_size=20)
plot_fixation_duration_for_category('STAI-T', 20, window_size=20)
plot_fixation_duration_for_category('STAI-S', 20, window_size=20)
plot_fixation_duration_for_category('BFI', 10, window_size=20)
plot_fixation_duration_for_category('FQ', 24, window_size=20)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

DATA = '../data/individual/processed'
PSY = '../data/individual/psychometric'

sed_data = pd.read_csv(f'{DATA}/sed_fix_03.csv')
sed_data['datetime'] = pd.to_datetime(sed_data['datetime'], format='%Y/%m/%d %H:%M:%S.%f')

psychometric_data = pd.read_csv(f'{PSY}/Psychometric_Test_Results_03.csv')
psychometric_data['Question Start Time'] = pd.to_datetime(psychometric_data['Question Start Time'], utc=True).dt.tz_convert(None)
psychometric_data['Question Answer Time'] = pd.to_datetime(psychometric_data['Question Answer Time'], utc=True).dt.tz_convert(None)

def plot_fixation_duration_for_category(category_name, expected_questions, window_size=20):
    category_data = psychometric_data[psychometric_data['Type'] == category_name].copy()

    segments = []
    question_times = []
    for _, row in category_data.iterrows():
        mask = (sed_data['datetime'] >= row['Question Start Time']) & (sed_data['datetime'] <= row['Question Answer Time'])
        seg = sed_data.loc[mask]
        if not seg.empty:
            segments.append(seg)
        question_times.append(row['Question Answer Time'])

    category_eye_data = pd.concat(segments, ignore_index=True) if segments else pd.DataFrame()
    if len(question_times) != expected_questions:
        print(f"Warning: {category_name} has {len(question_times)} questions, expected {expected_questions}")

    category_eye_data = category_eye_data.sort_values(by='datetime').reset_index(drop=True)
    category_eye_data['smoothed_fixation_duration'] = category_eye_data['duration'].rolling(window=window_size).mean()
    category_eye_data = category_eye_data[np.isfinite(category_eye_data['smoothed_fixation_duration'])]

    plt.figure(figsize=(12, 6))
    plt.plot(category_eye_data['datetime'], category_eye_data['smoothed_fixation_duration'], label='Fixation Duration', color='b')

    for i, time in enumerate(question_times, start=1):
        if not category_eye_data.empty:
            time_diffs = (category_eye_data['datetime'] - time).abs()
            nearest_idx = time_diffs.idxmin()
            fd = category_eye_data.loc[nearest_idx, 'smoothed_fixation_duration']
            plt.scatter(category_eye_data.loc[nearest_idx, 'datetime'], fd, color='red', s=50, zorder=5)
            plt.text(category_eye_data.loc[nearest_idx, 'datetime'], fd + 0.1, f'Q{i}', fontsize=9, rotation=45, ha='right')

    plt.xlabel('Time')
    plt.ylabel('Fixation Duration')
    plt.title(f'Fixation Duration Changes During {category_name}')
    plt.xticks(rotation=45)
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    plt.close()

plot_fixation_duration_for_category('HADS', 14, window_size=20)
plot_fixation_duration_for_category('STAI-T', 20, window_size=20)
plot_fixation_duration_for_category('STAI-S', 20, window_size=20)
plot_fixation_duration_for_category('BFI', 10, window_size=20)
plot_fixation_duration_for_category('FQ', 24, window_size=20)